# Metric view smoke test

Runs after `register_each_mv`. For every metric view in the manifest, queries every measure once: `SELECT MEASURE(`m1`), MEASURE(`m2`), ... FROM <catalog>.<schema>.<mv> LIMIT 1` (a dimension-only view selects its first dimension). Fails the task if any view does not resolve or a query errors.

**Widgets:** `bundle_root`, `metric_views_dir`, `target_catalog`, `target_schema` (all required).

In [0]:
%pip install --upgrade databricks-sdk pyyaml
dbutils.library.restartPython()

In [0]:
WIDGETS = ["bundle_root", "metric_views_dir", "target_catalog", "target_schema"]
for w in WIDGETS:
    dbutils.widgets.text(w, "")
args = {w: dbutils.widgets.get(w).strip() for w in WIDGETS}

missing = [k for k, v in args.items() if not v]
if missing:
    raise ValueError(f"Required widget(s) not set: {missing}")
for k, v in args.items():
    print(f"{k}={v}")

In [0]:
import importlib
import json
import re
import sys
import types
from pathlib import Path


def load_sdk(root: str):
    """Import mv_yaml and manifest from <root>/src/genie_ontology_modeled_knowledge_sdk_vendored/.

    The vendored files are verbatim copies of src/genie_ontology_modeled_knowledge_sdk/ (they import
    each other as genie_ontology_modeled_knowledge_sdk.*), so the vendored folder is registered as
    the genie_ontology_modeled_knowledge_sdk package. Falls back to an installed genie_ontology_modeled_knowledge_sdk.
    """
    vendored = Path(root) / "src" / "genie_ontology_modeled_knowledge_sdk_vendored"
    if (vendored / "mv_yaml.py").is_file():
        pkg = types.ModuleType("genie_ontology_modeled_knowledge_sdk")
        pkg.__path__ = [str(vendored)]
        sys.modules["genie_ontology_modeled_knowledge_sdk"] = pkg
    try:
        return (importlib.import_module("genie_ontology_modeled_knowledge_sdk.mv_yaml"),
                importlib.import_module("genie_ontology_modeled_knowledge_sdk.manifest"))
    except ImportError as exc:
        raise ImportError(
            f"genie_ontology_modeled_knowledge_sdk not found: expected the vendored library in {vendored} "
            "(scaffold_project copies it in; see src/genie_ontology_modeled_knowledge_sdk_vendored/README.md)"
        ) from exc


mv_yaml, manifest = load_sdk(args["bundle_root"])

In [0]:
_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def quote(name: str) -> str:
    return name if _IDENT.match(name) else "`" + name.replace("`", "``") + "`"


root = Path(args["bundle_root"])
entries = manifest.build_manifest(root / args["metric_views_dir"], bundle_root=root)
if not entries:
    raise ValueError("No metric views in the manifest; nothing to smoke test")

prefix = f"{quote(args['target_catalog'])}.{quote(args['target_schema'])}"
results = []
for e in entries:
    doc = mv_yaml.load_mv(root / e["yaml_path"])
    measures = [m["name"] for m in (doc.get("measures") or []) if isinstance(m, dict) and m.get("name")]
    dims = [d["name"] for d in (doc.get("dimensions") or doc.get("fields") or []) if isinstance(d, dict) and d.get("name")]
    if measures:
        select = ", ".join(f"MEASURE({quote(m)})" for m in measures)
    elif dims:
        select = quote(dims[0])
    else:
        results.append({"name": e["name"], "ok": False, "error": "no measures or dimensions"})
        continue
    sql = f"SELECT {select} FROM {prefix}.{quote(e['name'])} LIMIT 1"
    try:
        rows = spark.sql(sql).collect()
        results.append({"name": e["name"], "ok": True, "measures": len(measures), "rows": len(rows)})
        print(f"PASS {e['name']}: {len(measures)} measure(s)")
    except Exception as exc:  # recorded and re-raised below as one failure
        msg = str(exc).splitlines()[0][:500]
        results.append({"name": e["name"], "ok": False, "error": msg, "sql": sql})
        print(f"FAIL {e['name']}: {msg}")

In [0]:
failed = [r for r in results if not r["ok"]]
summary = {"metric_views": len(results), "passed": len(results) - len(failed), "failed": [r["name"] for r in failed]}
dbutils.jobs.taskValues.set(key="summary", value=summary)
print(json.dumps(summary, indent=2))
if failed:
    raise AssertionError(f"Smoke test failed for {len(failed)} metric view(s): " + json.dumps(failed)[:4000])
dbutils.notebook.exit(json.dumps(summary))